In [1]:
from __future__ import print_function
import numpy as np
import pandas as pd

class TreeNode(object):
    def __init__(self, ids = None, children = [], entropy = 0, depth = 0):
        self.ids = ids           # index of data in this node
        self.entropy = entropy   # entropy, will fill later
        self.depth = depth       # distance to root node
        self.split_attribute = None # which attribute is chosen, it non-leaf
        self.children = children # list of its child nodes
        self.order = None       # order of values of split_attribute in children
        self.label = None       # label of node if it is a leaf

    def set_properties(self, split_attribute, order):
        self.split_attribute = split_attribute # split at which attribute
        self.order = order # order of this node's children

    def set_label(self, label):
        self.label = label # set label if the node is a leaf

In [2]:
def entropy(freq):
    # remove prob 0
    freq_0 = freq[np.array(freq).nonzero()[0]]
    prob_0 = freq_0/float(freq_0.sum())
    return -np.sum(prob_0*np.log(prob_0))

In [3]:
class DecisionTreeID3(object):
    def __init__(self, max_depth= 10, min_samples_split = 2, min_gain = 1e-4):
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.Ntrain = 0
        self.min_gain = min_gain

    def fit(self, data, target):
        self.Ntrain = data.count()[0]
        self.data = data
        self.attributes = list(data)
        self.target = target
        self.labels = target.unique()

        ids = range(self.Ntrain)
        self.root = TreeNode(ids = ids, entropy = self._entropy(ids), depth = 0)
        queue = [self.root]
        while queue:
            node = queue.pop()
            if node.depth < self.max_depth or node.entropy < self.min_gain:
                node.children = self._split(node)
                if not node.children: #leaf node
                    self._set_label(node)
                queue += node.children
            else:
                self._set_label(node)

    def _entropy(self, ids):
        # calculate entropy of a node with index ids
        if len(ids) == 0: return 0
        ids = [i+1 for i in ids] # panda series index starts from 1
        freq = np.array(self.target[ids].value_counts())
        return entropy(freq)

    def _set_label(self, node):
        # find label for a node if it is a leaf
        # simply chose by major voting
        target_ids = [i + 1 for i in node.ids]  # target is a series variable
        node.set_label(self.target[target_ids].mode()[0]) # most frequent label

    def _split(self, node):
        ids = node.ids
        best_gain = 0
        best_splits = []
        best_attribute = None
        order = None
        sub_data = self.data.iloc[ids, :]
        for i, att in enumerate(self.attributes):
            values = self.data.iloc[ids, i].unique().tolist()
            if len(values) == 1: continue # entropy = 0
            splits = []
            for val in values:
                sub_ids = sub_data.index[sub_data[att] == val].tolist()
                splits.append([sub_id-1 for sub_id in sub_ids])
            # don't split if a node has too small number of points
            if min(map(len, splits)) < self.min_samples_split: continue
            # information gain
            HxS= 0
            for split in splits:
                HxS += len(split)*self._entropy(split)/len(ids)
            gain = node.entropy - HxS
            if gain < self.min_gain: continue # stop if small gain
            if gain > best_gain:
                best_gain = gain
                best_splits = splits
                best_attribute = att
                order = values
        node.set_properties(best_attribute, order)
        child_nodes = [TreeNode(ids = split,
                     entropy = self._entropy(split), depth = node.depth + 1) for split in best_splits]
        return child_nodes

    def predict(self, new_data):
        """
        :param new_data: a new dataframe, each row is a datapoint
        :return: predicted labels for each row
        """
        npoints = new_data.count()[0]
        labels = [None]*npoints
        for n in range(npoints):
            x = new_data.iloc[n, :] # one point
            # start from root and recursively travel if not meet a leaf
            node = self.root
            while node.children:
                node = node.children[node.order.index(x[node.split_attribute])]
            labels[n] = node.label

        return labels

In [ ]:
from pathlib import Path

DATA_DIR = Path("data") if Path("data").is_dir() else Path("04_id3") / "data"

def read_id3_csv(csv_path):
    """??c CSV UTF-8/UTF-8-SIG v? ??t index 1-based theo ID3 g?c."""
    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="utf-8")
    return df

def print_tree_rules(node, conditions=None):
    """In c?y th?nh c?c lu?t t? g?c ??n t?ng n?t l?."""
    conditions = conditions or []
    if not node.children:
        rule = " AND ".join(conditions) if conditions else "TRUE"
        print(f"IF {rule} THEN {node.label}")
        return
    for value, child in zip(node.order, node.children):
        print_tree_rules(child, conditions + [f"{node.split_attribute} = {value}"])

def run_id3(csv_path, target_col, id_cols=(), max_depth=3, min_samples_split=2):
    """??c m?t CSV, hu?n luy?n ID3 v? in d? li?u, d? ?o?n, c?y/lu?t."""
    df = read_id3_csv(csv_path)
    if target_col not in df.columns:
        raise ValueError(f"Kh?ng t?m th?y c?t nh?n {target_col!r}; c?c c?t: {list(df.columns)}")
    ignored = [c for c in id_cols if c in df.columns]
    X = df.drop(columns=[target_col, *ignored]).copy()
    y = df[target_col].copy()
    if X.empty:
        raise ValueError("Kh?ng c?n thu?c t?nh ??u v?o sau khi b? c?t nh?n/ID.")
    # ID3 hi?n t?i chia theo gi? tr? r?i r?c; t? ch?i s? li?n t?c ?? tr?nh chia sai.
    numeric = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    if numeric:
        raise ValueError(f"ID3 hi?n t?i ch? h? tr? thu?c t?nh ph?n lo?i; c?n x? l? c?t li?n t?c: {numeric}")
    df.index = range(1, len(df) + 1)  # thu?t to?n ID3 g?c d?ng index b?t ??u t? 1
    X.index = df.index
    y.index = df.index
    tree = DecisionTreeID3(max_depth=max_depth, min_samples_split=min_samples_split)
    tree.fit(X, y)
    print("???ng d?n:", csv_path)
    print("K?ch th??c:", X.shape, "| c?t nh?n:", target_col)
    print("C?c c?t:", list(X.columns), "| ph?n b? nh?n:", y.value_counts().to_dict())
    print("D? li?u:")
    print(df.to_string(index=False))
    print("D? ?o?n tr?n d? li?u:", tree.predict(X))
    print("Lu?t quy?t ??nh:")
    print_tree_rules(tree.root)
    return X, y, tree


## Dataset 1: weather

Thu?c t?nh: outlook, temperature, humidity, wind. Nh?n: play. B? c?t ??nh danh id.

In [ ]:
# Kh?i ??c l?p: l?i t?i ??y ???c b?t ri?ng, kh?ng d?ng c?c dataset ph?a sau.
try:
    X_weather, y_weather, tree_weather = run_id3(
        DATA_DIR / "weather.csv", target_col="play", id_cols=['id'],
        max_depth=3, min_samples_split=2
    )
except Exception as exc:
    print("L?i khi ch?y weather.csv:", repr(exc))


## Dataset 2: buys_computer

Thu?c t?nh: age, income, student, credit_rating. Nh?n: buys_computer.

In [ ]:
# Kh?i ??c l?p: l?i t?i ??y ???c b?t ri?ng, kh?ng d?ng c?c dataset ph?a sau.
try:
    X_buys_computer, y_buys_computer, tree_buys_computer = run_id3(
        DATA_DIR / "buys_computer.csv", target_col="buys_computer", id_cols=[],
        max_depth=3, min_samples_split=2
    )
except Exception as exc:
    print("L?i khi ch?y buys_computer.csv:", repr(exc))


## Dataset 3: credit_risk

Thu?c t?nh: ?? tu?i, H?n nh?n, S? h?u B?S, Thu nh?p. Nh?n: R?i ro t?n d?ng. B? c?t ??nh danh ID.

In [ ]:
# Kh?i ??c l?p: l?i t?i ??y ???c b?t ri?ng, kh?ng d?ng c?c dataset ph?a sau.
try:
    X_credit_risk, y_credit_risk, tree_credit_risk = run_id3(
        DATA_DIR / "credit_risk.csv", target_col="R?i ro t?n d?ng", id_cols=['ID'],
        max_depth=3, min_samples_split=2
    )
except Exception as exc:
    print("L?i khi ch?y credit_risk.csv:", repr(exc))


## Animation ID3 (weather)

Minh h?a t?ng b??c t?ch c?y tr?n weather; ph?n n?y ch?y ??c l?p v?i c?c kh?i hu?n luy?n ? tr?n.

In [7]:
# ===== Cell biểu diễn animation ID3 =====
# Cần có sẵn: DecisionTreeID3, TreeNode, entropy, và X (DataFrame thuộc tính, index từ 1), y (Series nhãn)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


class RecordingID3(DecisionTreeID3):
    """DecisionTreeID3 + ghi lại từng bước xử lý nút để vẽ animation."""

    def fit(self, data, target):
        self.Ntrain = len(data)
        self.data, self.target = data, target
        self.attributes = list(data)
        self.labels = target.unique()
        self.steps = []                                  # (node, gains, best_attribute)
        ids = range(self.Ntrain)
        self.root = TreeNode(ids=ids, entropy=self._entropy(ids), depth=0)
        queue = [self.root]
        while queue:
            node = queue.pop()
            self.gains = {}
            if node.depth < self.max_depth or node.entropy < self.min_gain:
                node.children = self._split(node)
                if not node.children:
                    self._set_label(node)
                queue += node.children
            else:
                self._set_label(node)
            self.steps.append((node, dict(self.gains), node.split_attribute))

    def _split(self, node):
        ids = node.ids
        best_gain, best_splits, best_attribute, order = 0, [], None, None
        sub_data = self.data.iloc[ids, :]
        for i, att in enumerate(self.attributes):
            values = self.data.iloc[ids, i].unique().tolist()
            if len(values) == 1:
                continue
            splits = []
            for val in values:
                sub_ids = sub_data.index[sub_data[att] == val].tolist()
                splits.append([sub_id - 1 for sub_id in sub_ids])
            if min(map(len, splits)) < self.min_samples_split:
                continue
            HxS = sum(len(s) * self._entropy(s) / len(ids) for s in splits)
            gain = node.entropy - HxS
            self.gains[att] = gain                       # ghi lại để vẽ
            if gain < self.min_gain:
                continue
            if gain > best_gain:
                best_gain, best_splits, best_attribute, order = gain, splits, att, values
        node.set_properties(best_attribute, order)
        return [TreeNode(ids=s, entropy=self._entropy(s), depth=node.depth + 1)
                for s in best_splits]


def animate_id3(X, y, max_depth=3, min_samples_split=2, interval=1800):
    tree = RecordingID3(max_depth=max_depth, min_samples_split=min_samples_split)
    tree.fit(X, y)
    steps = tree.steps

    # --- layout: lá xếp đều, nút cha nằm giữa các con ---
    pos, counter = {}, [0]
    def rec(n):
        if not n.children:
            pos[id(n)] = (counter[0], -n.depth); counter[0] += 1
        else:
            for c in n.children: rec(c)
            xs = [pos[id(c)][0] for c in n.children]
            pos[id(n)] = ((xs[0] + xs[-1]) / 2, -n.depth)
    rec(tree.root)

    palette = plt.cm.Set2(np.linspace(0, 1, max(len(tree.labels), 3)))
    label_color = {l: palette[i] for i, l in enumerate(tree.labels)}
    max_gain = max([g for _, gs, _ in steps for g in gs.values()] + [0.1])
    max_depth_seen = max(n.depth for n, _, _ in steps)

    def counts(n):
        vc = tree.target.iloc[list(n.ids)].value_counts()
        return ", ".join(f"{vc[l]} {l}" for l in tree.labels if l in vc)

    fig, (ax, axg) = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [2.2, 1]})
    fig.subplots_adjust(left=0.02, right=0.97, top=0.91, bottom=0.12, wspace=0.25)

    def draw(k):
        node_now, gains, best = steps[k]
        processed = {id(s[0]) for s in steps[:k + 1]}
        visible = {id(tree.root)}
        for s in steps[:k + 1]:
            visible |= {id(c) for c in s[0].children}

        ax.clear(); ax.set_axis_off()
        ax.set_xlim(-0.7, max(p[0] for p in pos.values()) + 0.7)
        ax.set_ylim(-max_depth_seen - 0.6, 0.6)

        def draw_edges(n):
            for j, c in enumerate(n.children):
                if id(c) not in visible: continue
                (x0, y0), (x1, y1) = pos[id(n)], pos[id(c)]
                ax.plot([x0, x1], [y0 - 0.12, y1 + 0.12], color="gray", lw=1.5, zorder=1)
                ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.04, str(n.order[j]), ha="center",
                        fontsize=9, color="dimgray", bbox=dict(fc="white", ec="none", pad=1))
                draw_edges(c)
        draw_edges(tree.root)

        def draw_nodes(n):
            if id(n) not in visible: return
            x, yy = pos[id(n)]
            done = id(n) in processed
            is_leaf = done and n.label is not None
            fc = label_color[n.label] if is_leaf else "white"
            ec = "crimson" if n is node_now else ("teal" if done else "gray")
            title = str(n.label) if is_leaf else (str(n.split_attribute) if done else "?")
            ax.text(x, yy, f"{title}\n{counts(n)}\nH={abs(n.entropy):.3f}", ha="center",
                    va="center", fontsize=9, zorder=3,
                    bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec=ec,
                              lw=3 if n is node_now else 1.5))
            for c in n.children: draw_nodes(c)
        draw_nodes(tree.root)
        ax.set_title(f"ID3 – bước {k + 1}/{len(steps)}", fontsize=13)

        axg.clear()
        axg.set_title(f"Information gain tại nút đang xét (H={abs(node_now.entropy):.3f})", fontsize=11)
        names = [str(a) for a in tree.attributes]
        vals = [gains.get(a, 0) for a in tree.attributes]
        cols = ["teal" if a == best else ("lightgray" if a not in gains else "steelblue")
                for a in tree.attributes]
        axg.barh(names, vals, color=cols)
        for i, a in enumerate(tree.attributes):
            axg.text(vals[i] + 0.005, i, f"{gains[a]:.3f}" if a in gains else "bỏ qua",
                     va="center", fontsize=9)
        axg.set_xlim(0, max_gain * 1.3)
        axg.invert_yaxis()
        if not gains:
            axg.text(0.5, 0.5, "Không cần tách\n(nút thuần nhất hoặc đạt max_depth)",
                     transform=axg.transAxes, ha="center", va="center", fontsize=10, color="gray")
        axg.set_xlabel(f"Chọn '{best}' (gain lớn nhất)" if best
                       else f"Nút lá → nhãn '{node_now.label}'", fontsize=11, color="crimson")

    anim = FuncAnimation(fig, draw, frames=len(steps), interval=interval, repeat=True)
    plt.close(fig)                      # tránh hiện thêm ảnh tĩnh trong Colab
    return anim


anim = animate_id3(X, y, max_depth=3, min_samples_split=2)
HTML(anim.to_jshtml())                  # hiện animation có nút play/pause
anim.save("id3_animation.gif", writer="pillow", fps=0.1)